# Gold: das Sternschema für dynamische Abfragen

**Zweck:** Dieses Notebook belegt zwei Entscheidungen der Gold-Schicht und
rechnet beide nach.

1. **Warum Gold nicht aggregiert.** Der ursprüngliche Entwurf sah zwei
   vorberechnete Würfel vor. Gemessen bringen sie nichts — und kosten genau
   die Beweglichkeit, für die die Auswertungsplattform da ist.
2. **Warum Gold trotzdem eine andere Granularität hat als Silver.** Detroits Zeilen
   sind Anklagen, Chicagos sind Delikte; ohne Auswahl zählt Detroit 3,7 %
   zu hoch.
3. **Wie Tracts einer Stadt zugeordnet werden.** Das ist der eigentliche Grund
   für diese Schicht: Ohne Stadtgrenze gibt es keinen Nenner, und damit keine
   Rate je Einwohner.

**Stand:** 20. September 2026. Alle Zahlen im Fliesstext stammen aus den Zellen
darunter und werden bei jedem Lauf neu gerechnet — wer das Notebook ausführt,
sieht sofort, wenn eine nicht mehr stimmt.

In [ ]:
import sys
import time
from pathlib import Path

import duckdb

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'config').exists():
    raise RuntimeError('Projekt-Root mit config/ nicht gefunden')
if str(PROJECT_ROOT / 'src' / 'crime') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'crime'))

from serving.release_reader import approved_root
DATA_ROOT = approved_root(PROJECT_ROOT / 'data')
print(f"Analyse auf festgehaltenem Datenstand: {DATA_ROOT}")

from storage.duckdb import verbinde
con = verbinde()
con.execute("SET TimeZone='UTC'")

# Aus dem Register, nicht von Hand: Eine siebte Sterntabelle erscheint hier
# von selbst. Seit Vertrag 6 sind es sechs - dim_tract_acs ist dazugekommen.
from gold_pipeline import TABELLEN as GOLD, einwohner_je_stadt

# Der Nenner jeder Rate, einmal gebaut: ein Verbund ueber geoid UND vintage.
# Die Regel steht in gold_pipeline, damit sie nicht an neun Stellen gepflegt
# werden muss - und damit niemand das zweite Praedikat vergisst. Ohne es
# summiert die Abfrage jeden Tract so oft, wie Vintages geladen sind.
NENNER = einwohner_je_stadt('dim_tract', 'dim_tract_acs')
fehlend = []
for name in GOLD:
    pfad = DATA_ROOT / 'gold' / f'dataset={name}' / 'part-0001.parquet'
    if pfad.exists():
        con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{pfad}')")
    else:
        fehlend.append(name)

_silber = DATA_ROOT / 'silver' / 'dataset=crime_offense' / 'part-0001.parquet'
SILBER = f"read_parquet('{_silber}')" if _silber.exists() else None

if fehlend:
    print('Gold fehlt:', fehlend)
    print('Zuerst:  python -m gold_pipeline.build_gold')
else:
    print(f"{'Tabelle':16}{'Zeilen':>12}")
    for name in GOLD:
        n = con.execute(f'SELECT count(*) FROM {name}').fetchone()[0]
        print(f'{name:16}{n:>12,}')


def miss(sql, laeufe=5):
    """Kleinste gemessene Laufzeit in Millisekunden, plus Zeilenzahl."""
    con.execute(sql).fetchall()
    zeiten = []
    for _ in range(laeufe):
        start = time.perf_counter()
        zeilen = con.execute(sql).fetchall()
        zeiten.append((time.perf_counter() - start) * 1000)
    return min(zeiten), len(zeilen)

## Zwei Bauweisen — und warum die Wahl hier gemessen wurde

Bevor die Messung kommt, die Begriffe. Beide Bauweisen beantworten dieselbe
Frage — *wie viele Delikte, wo, wann, welcher Art* — und sie tun es auf
entgegengesetzte Weise.

### Der Würfel: vorher zählen

Ein **Würfel** (OLAP-Cube) ist eine Tabelle, in der das Zählen schon passiert
ist. Man wählt die **Achsen** — hier Tract, Monat, Deliktfamilie — und die
**Feinheit** jeder Achse, und rechnet dann für jede Kombination die Zahl aus.

<figure>
<svg viewBox="0 0 830 372" role="img" width="100%" height="auto"
     aria-label="Ein vorberechneter Würfel mit den Achsen Tract, Monat und Deliktfamilie. Jede Zelle hält eine fertige Zahl; der Tag und das einzelne Delikt sind darin nicht mehr enthalten.">
  <g stroke="currentColor" stroke-width="1.3" stroke-linejoin="round" fill="none">
    <polygon points="236,96 356,165 476,96 356,27" fill="currentColor" fill-opacity="0.05"/>
    <polygon points="236,96 356,165 356,297 236,228" fill="currentColor" fill-opacity="0.12"/>
    <polygon points="356,165 476,96 476,228 356,297" fill="currentColor" fill-opacity="0.02"/>
    <g stroke-opacity="0.3" stroke-width="1">
      <line x1="276" y1="119" x2="396" y2="50"/>
      <line x1="276" y1="73" x2="396" y2="142"/>
      <line x1="276" y1="119" x2="276" y2="251"/>
      <line x1="236" y1="140" x2="356" y2="209"/>
      <line x1="396" y1="142" x2="396" y2="274"/>
      <line x1="356" y1="209" x2="476" y2="140"/>
      <line x1="316" y1="142" x2="436" y2="73"/>
      <line x1="316" y1="50" x2="436" y2="119"/>
      <line x1="316" y1="142" x2="316" y2="274"/>
      <line x1="236" y1="184" x2="356" y2="253"/>
      <line x1="436" y1="119" x2="436" y2="251"/>
      <line x1="356" y1="253" x2="476" y2="184"/>
    </g>
    <polygon points="396,186 436,163 436,207 396,230" fill="#c2410c" fill-opacity="0.92" stroke="none"/>
  </g>
  <g fill="currentColor" font-size="13" font-weight="600" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif">
    <text x="222.0" y="167.0" text-anchor="end">Monat</text>
    <text x="290.0" y="292.5" text-anchor="middle">Tract</text>
    <text x="426.0" y="292.5" text-anchor="middle">Deliktfamilie</text>
  </g>
  <g font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="12.5">
    <path d="M 416.0 196.5 L 540 168 L 566 168" stroke="currentColor" stroke-width="1" stroke-opacity="0.45" fill="none"/>
    <circle cx="566" cy="168" r="2.5" fill="currentColor"/>
    <text x="578" y="158" fill="currentColor" font-weight="600">eine Zelle = eine fertige Zahl</text>
    <text x="578" y="177" fill="currentColor" fill-opacity="0.7">412 Diebstähle</text>
    <text x="578" y="194" fill="currentColor" fill-opacity="0.7">Tract 170318391 · März · Larceny</text>
  </g>
  <g font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="12.5" fill="currentColor" fill-opacity="0.7">
    <text x="14" y="350">Nicht mehr enthalten: der Tag, das einzelne Delikt, der einzelne Fall.</text>
  </g>
</svg>
<figcaption><strong>Der Würfel.</strong> Achsen und Feinheit werden <em>vor</em> der ersten Frage gewählt. Danach steht in jeder Zelle eine fertige Zahl — und nur noch sie.</figcaption>
</figure>

Der Gewinn ist offensichtlich: Die Frage *„wie viele Diebstähle im März in
Tract 170318391"* ist ein Nachschlagen, keine Rechnung. Der Preis steht in der
Zeile unter der Grafik — was in der Zelle zusammengefasst wurde, ist weg. Nicht
versteckt, nicht langsam: **weg**. Kein Filter bringt es zurück.

Zwei Regeln folgen daraus, und beide sind hart:

* **Nach unten geht nichts.** Ein Monatswürfel kennt keinen 9. November. Wer
  den Tag braucht, braucht einen anderen Würfel.
* **Was nicht als Achse gewählt wurde, existiert nicht.** `fact_city_day` hat
  keine Tract-Achse — die Frage *„in welchem Tract"* ist dort nicht schwer zu
  beantworten, sondern unmöglich.

Deshalb hätte der ursprüngliche Entwurf auch **zwei** Würfel gebraucht: einen
mit Tract ohne Tag, einen mit Tag ohne Tract. Jede weitere Frage hätte einen
dritten verlangt.

### Das Sternschema: die Granularität behalten, schnell fragen

Ein **Sternschema** dreht das um. In der Mitte steht **eine** Faktentabelle im
feinstmöglichen Granularität — bei uns eine Zeile je Tat. Sie trägt nur **Schlüssel**
und **Messgrössen**, sonst nichts. Alles Beschreibende liegt aussen, in kleinen
**Dimensionstabellen**.

<figure>
<svg viewBox="0 0 800 452" role="img" width="100%" height="auto"
     aria-label="Das Sternschema: die Faktentabelle fact_offense in der Mitte, darum die fünf Dimensionen dim_tract, dim_tract_acs, dim_offense, dim_date und dim_city_day, je über einen Schlüssel verbunden. dim_tract_acs hängt an zwei Spalten, weil die Einwohnerzahl je ACS-Vintage gilt.">
  <defs><marker id="pfeil" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
    <path d="M 0 1 L 9 5 L 0 9 z" fill="currentColor"/></marker></defs>
  <g>
    <rect x="310" y="26" width="190" height="70" rx="5" fill="currentColor" fill-opacity="0.045" stroke="currentColor" stroke-width="1.1"/>
    <text x="323" y="50" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="13.5" font-weight="600" fill="currentColor">dim_date</text>
    <text x="323" y="68" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="11.5" fill="currentColor" fill-opacity="0.78">366 Zeilen · ein Kalendertag</text>
    <text x="323" y="85" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="10.5" fill="currentColor" fill-opacity="0.55">month · is_weekend</text>
  </g>
  <g>
    <rect x="310" y="356" width="190" height="70" rx="5" fill="currentColor" fill-opacity="0.045" stroke="currentColor" stroke-width="1.1"/>
    <text x="323" y="380" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="13.5" font-weight="600" fill="currentColor">dim_city_day</text>
    <text x="323" y="398" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="11.5" fill="currentColor" fill-opacity="0.78">732 Zeilen · Stadt und Tag</text>
    <text x="323" y="415" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="10.5" fill="currentColor" fill-opacity="0.55">temp_max_c · snowfall_mm</text>
  </g>
  <g>
    <rect x="14" y="191" width="190" height="70" rx="5" fill="currentColor" fill-opacity="0.045" stroke="currentColor" stroke-width="1.1"/>
    <text x="27" y="215" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="13.5" font-weight="600" fill="currentColor">dim_tract</text>
    <text x="27" y="233" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="11.5" fill="currentColor" fill-opacity="0.78">6 282 Zeilen · ein Tract</text>
    <text x="27" y="250" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="10.5" fill="currentColor" fill-opacity="0.55">city · reference_vintage</text>
  </g>
  <g>
    <rect x="596" y="191" width="190" height="70" rx="5" fill="currentColor" fill-opacity="0.045" stroke="currentColor" stroke-width="1.1"/>
    <text x="609" y="215" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="13.5" font-weight="600" fill="currentColor">dim_offense</text>
    <text x="609" y="233" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="11.5" fill="currentColor" fill-opacity="0.78">85 Zeilen · ein NIBRS-Delikt</text>
    <text x="609" y="250" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="10.5" fill="currentColor" fill-opacity="0.55">nibrs_category · group</text>
  </g>
  <g>
    <rect x="14" y="296" width="190" height="70" rx="5" fill="currentColor" fill-opacity="0.045" stroke="currentColor" stroke-width="1.1"/>
    <text x="27" y="320" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="13.5" font-weight="600" fill="currentColor">dim_tract_acs</text>
    <text x="27" y="338" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="11.5" fill="currentColor" fill-opacity="0.78">6 282 je Vintage · Tract × Vintage</text>
    <text x="27" y="355" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="10.5" fill="currentColor" fill-opacity="0.55">population · poverty_rate</text>
  </g>
  <g>
    <rect x="300" y="182" width="200" height="88" rx="5" fill="#c2410c" fill-opacity="0.10" stroke="currentColor" stroke-width="1.7"/>
    <text x="313" y="206" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="13.5" font-weight="600" fill="currentColor">fact_offense</text>
    <text x="313" y="224" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="11.5" fill="currentColor" fill-opacity="0.78">345 073 Zeilen · eine Tat</text>
    <text x="313" y="241" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="10.5" fill="currentColor" fill-opacity="0.55">Schlüssel + Messgrössen</text>
  </g>
  <line x1="400" y1="96" x2="400" y2="178" stroke="currentColor" stroke-width="1.2" stroke-opacity="0.55" marker-end="url(#pfeil)"/>
  <line x1="400" y1="356" x2="400" y2="274" stroke="currentColor" stroke-width="1.2" stroke-opacity="0.55" marker-end="url(#pfeil)"/>
  <line x1="204" y1="226" x2="296" y2="226" stroke="currentColor" stroke-width="1.2" stroke-opacity="0.55" marker-end="url(#pfeil)"/>
  <line x1="596" y1="226" x2="504" y2="226" stroke="currentColor" stroke-width="1.2" stroke-opacity="0.55" marker-end="url(#pfeil)"/>
  <line x1="204" y1="331" x2="298" y2="272" stroke="currentColor" stroke-width="1.2" stroke-opacity="0.55" marker-end="url(#pfeil)"/>
  <text x="212" y="318" text-anchor="start" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="10.5" fill="currentColor" fill-opacity="0.75">(tract_geoid, population_vintage)</text>
  <text x="408" y="142" text-anchor="start" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="11" fill="currentColor" fill-opacity="0.75">date</text>
  <text x="408" y="318" text-anchor="start" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="11" fill="currentColor" fill-opacity="0.75">(reporting_city, date)</text>
  <text x="250" y="218" text-anchor="middle" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="11" fill="currentColor" fill-opacity="0.75">tract_geoid</text>
  <text x="550" y="218" text-anchor="middle" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="11" fill="currentColor" fill-opacity="0.75">nibrs_code</text>
  <text x="14" y="446" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="12.5" fill="currentColor" fill-opacity="0.7">Jede Frage ist ein Filter, ein GROUP BY und ein JOIN — die Dimensionen tragen die Auswahllisten gleich mit.</text>
</svg>
<figcaption><strong>Das Sternschema.</strong> Die Faktentabelle trägt Schlüssel und Messgrössen, die Dimensionen die Bedeutung. Jede Kante ist ein JOIN-Schlüssel.</figcaption>
</figure>

Der Name kommt vom Bild: eine Mitte, Strahlen nach aussen. Drei Dinge daran
sind der eigentliche Punkt.

**Die Faktentabelle bleibt schlank.** Stünde in jeder ihrer 345 091 Zeilen der
Text `Larceny/Theft Offenses`, läge derselbe Satz zehntausendfach auf der
Platte. Statt dessen steht dort `23C` — drei Zeichen — und die Bedeutung genau
einmal in `dim_offense`.

**Die Dimensionen tragen die Auswahllisten.** Eine Oberfläche, die „Delikt
wählen" anbieten will, liest die 85 Zeilen aus `dim_offense`, nicht 345 091
Faktenzeilen. Dasselbe für Städte, Tage, Tracts.

**`dim_date` wird erzeugt, nicht abgeleitet.** Ein Tag ohne Delikt hat keine
Faktenzeile. Käme die Zeitachse aus den Fakten, fehlte dieser Tag — und eine
Zeitreihe zeigte eine Lücke, wo eine Null stehen müsste. Die Dimension kennt
alle 366 Tage von 2024, ob etwas passiert ist oder nicht.

**`dim_tract_acs` hängt an zwei Spalten — und das ist der Grund, warum es sie
gibt.** Die Einwohnerzahl ist keine Eigenschaft eines Tracts, sondern einer
*Schätzung für ein Jahr*. Solange nur 2024 gebaut wurde, fiel das nicht auf:
Eine Zeile je Tract konnte die eine Zahl tragen, und `dim_tract` trug sie. Über
mehrere Jahrgänge geht das nicht mehr — eine Tabelle mit einer Zeile je Tract
kann nur **eine** Einwohnerzahl halten, und jede Tat bekäme denselben Nenner,
auch die von 2021. Die Masse liegen deshalb je Vintage in `dim_tract_acs`, und
`fact_offense.population_vintage` sagt je Zeile, welches gilt. Verbunden wird
über **beide** Spalten. Wer nur `tract_geoid` verbindet, bekommt je Tract so
viele Zeilen, wie Vintages geladen sind — und die Summe sieht aus wie eine Zahl.

Ab 2025 gibt es kein eigenes Vintage mehr: Das jüngste verfügbare ist 2024.
Eine Rate für 2025 ruht damit auf der Bevölkerung von 2024, und `population_vintage`
sagt das — in den Daten, nicht in einer Fussnote.

### Was die Wahl kostet — in Zeilen

<figure>
<svg viewBox="0 0 860 322" role="img" width="100%" height="auto"
     aria-label="Vier Granularitäten im Vergleich: Silver hält 348329 Anklagen, Gold 345091 Taten, die verworfenen Würfel nur noch 95243 beziehungsweise 11156 vorberechnete Zellen.">
  <text x="184" y="58" text-anchor="end" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="12.5" font-weight="600" fill="currentColor">silver/crime_offense</text>
  <text x="184" y="73" text-anchor="end" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="11" fill="currentColor" fill-opacity="0.62">eine Anklage</text>
  <rect x="196" y="44" width="376.0" height="32" rx="3" fill="#c2410c" fill-opacity="0.82" stroke="none"/>
  <text x="584.0" y="59" font-family="ui-monospace SFMono-Regular Menlo monospace" font-size="12" fill="currentColor" font-weight="600">348 311</text>
  <text x="584.0" y="74" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="10.5" fill="currentColor" fill-opacity="0.6">jede Anklage einzeln · 3,9 % Überzählung</text>
  <text x="184" y="120" text-anchor="end" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="12.5" font-weight="600" fill="currentColor">gold/fact_offense</text>
  <text x="184" y="135" text-anchor="end" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="11" fill="currentColor" fill-opacity="0.62">eine Tat</text>
  <rect x="196" y="106" width="372.4" height="32" rx="3" fill="#c2410c" fill-opacity="0.82" stroke="none"/>
  <text x="580.4" y="121" font-family="ui-monospace SFMono-Regular Menlo monospace" font-size="12" fill="currentColor" font-weight="600">345 073</text>
  <text x="580.4" y="136" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="10.5" fill="currentColor" fill-opacity="0.6">je Tat eine zählende Zeile · das gebaute Granularität</text>
  <text x="184" y="182" text-anchor="end" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="12.5" font-weight="600" fill="currentColor">fact_tract_month</text>
  <text x="184" y="197" text-anchor="end" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="11" fill="currentColor" fill-opacity="0.62">Tract × Monat × Familie</text>
  <rect x="196" y="168" width="102.9" height="32" rx="3" fill="currentColor" fill-opacity="0.07" stroke="currentColor" stroke-width="1.1" stroke-dasharray="4 3" stroke-opacity="0.55"/>
  <text x="310.9" y="183" font-family="ui-monospace SFMono-Regular Menlo monospace" font-size="12" fill="currentColor" font-weight="600">95 219</text>
  <text x="310.9" y="198" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="10.5" fill="currentColor" fill-opacity="0.6">kein Tag, kein einzelnes Delikt</text>
  <text x="184" y="244" text-anchor="end" font-family="ui-monospace,SFMono-Regular,Menlo,monospace" font-size="12.5" font-weight="600" fill="currentColor">fact_city_day</text>
  <text x="184" y="259" text-anchor="end" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="11" fill="currentColor" fill-opacity="0.62">Stadt × Tag × Familie</text>
  <rect x="196" y="230" width="12.1" height="32" rx="3" fill="currentColor" fill-opacity="0.07" stroke="currentColor" stroke-width="1.1" stroke-dasharray="4 3" stroke-opacity="0.55"/>
  <text x="220.1" y="245" font-family="ui-monospace SFMono-Regular Menlo monospace" font-size="12" fill="currentColor" font-weight="600">11 156</text>
  <text x="220.1" y="260" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="10.5" fill="currentColor" fill-opacity="0.6">kein Tract, kein einzelnes Delikt</text>
  <text x="196" y="26" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="11" font-weight="600" fill="#c2410c">gebaut</text>
  <text x="262" y="26" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="11" fill="currentColor" fill-opacity="0.6">· gestrichelt: der verworfene Entwurf</text>
  <text x="14" y="308" font-family="ui-sans-serif,system-ui,-apple-system,sans-serif" font-size="12.5" fill="currentColor" fill-opacity="0.7">Der Schritt von Silver zu Gold kostet 3 238 Zeilen — der Schritt zum Würfel 249 398.</text>
</svg>
<figcaption><strong>Was eine Zeile ist.</strong> Der Schritt von Silver zu Gold kostet 3 238 Zeilen, der Schritt zum Würfel 249 398 — und mit ihnen jede feinere Frage.</figcaption>
</figure>

Silver hält jede **Anklage**, Gold jede **Tat**. Das ist ein kleiner Schritt:
3 238 Zeilen, Detroits Mehrfachanklagen. Der Schritt zum Würfel ist ein
anderes Kaliber — 249 398 Zeilen, und mit ihnen jede Frage, die feiner ist als
Monat und Deliktfamilie.

Genau **diesen** Handel prüfen die nächsten beiden Abschnitte nach: Ist die
Zeitersparnis den Verlust wert? Die Antwort war nein, und sie war es deutlicher
als erwartet.

## 1. Die Messung, die den Entwurf geändert hat

Geplant waren `fact_tract_month` (`geoid` × Monat × Deliktfamilie) und
`fact_city_day` (Stadt × Tag × Familie). Die Begründung war Geschwindigkeit.

Die nächste Zelle misst dieselben Fragen **direkt auf der Silver-Tabelle** —
ohne jede Vorberechnung, ohne Index, ohne Würfel.

In [ ]:
if SILBER:
    fragen = [
        ('ein Delikt, ein Tag, eine Stadt', f"""
            SELECT count(*) FROM {SILBER}
            WHERE city='chicago' AND nibrs_code='23C'
              AND occurred_date_local=DATE '2024-11-09'"""),
        ('ein Delikt, ein Monat, beide Staedte', f"""
            SELECT city, count(*) FROM {SILBER}
            WHERE nibrs_code='240'
              AND date_trunc('month',occurred_date_local)=DATE '2024-06-01'
            GROUP BY 1"""),
        ('eine Deliktfamilie, je Monat und Stadt', f"""
            SELECT city, date_trunc('month',occurred_date_local) m, count(*)
            FROM {SILBER} WHERE nibrs_code LIKE '13%' GROUP BY 1,2"""),
        ('eine Stadt, alle Delikte, je Tag', f"""
            SELECT occurred_date_local, count(*) FROM {SILBER}
            WHERE city='detroit' GROUP BY 1"""),
        ('frei: Tract x Deliktfamilie', f"""
            SELECT tract_geoid, left(nibrs_code,2), count(*) FROM {SILBER}
            WHERE tract_geoid IS NOT NULL GROUP BY 1,2"""),
        ('die teuerste: alles x alles', f"""
            SELECT city, tract_geoid, occurred_date_local, nibrs_code, count(*)
            FROM {SILBER} GROUP BY 1,2,3,4"""),
    ]
    print(f"{'Abfrage auf Silver, ohne Vorberechnung':44}{'ms':>8}{'Zeilen':>10}")
    for name, sql in fragen:
        ms, zeilen = miss(sql)
        print(f'{name:44}{ms:>8.1f}{zeilen:>10,}')

**Der Engpass ist nicht die Datenmenge.** Jede realistische Frage liegt im
einstelligen Millisekundenbereich; selbst die pathologische Gruppierung über
alle vier Achsen bleibt unter einer Zehntelsekunde.

Die nächste Zelle prüft die Gegenrichtung: Spart eine **vorberechnete
Monatsspalte** überhaupt etwas?

In [ ]:
if SILBER:
    con.execute(f"""CREATE OR REPLACE TABLE mit_monat AS
                    SELECT *, date_trunc('month', occurred_date_local) AS monat
                    FROM {SILBER}""")
    a, _ = miss(f"""SELECT city, date_trunc('month',occurred_date_local), nibrs_code,
                    count(*) FROM {SILBER} GROUP BY 1,2,3""")
    b, _ = miss("""SELECT city, monat, nibrs_code, count(*)
                   FROM mit_monat GROUP BY 1,2,3""")
    print(f'date_trunc zur Laufzeit : {a:6.1f} ms')
    print(f'Spalte liegt schon da   : {b:6.1f} ms')
    print(f'Ersparnis               : {a - b:6.1f} ms')
    con.execute('DROP TABLE mit_monat')

## 2. Was die Aggregation gekostet hätte

Die Zeitersparnis ist also keine. Der Preis dagegen ist konkret: Ein Würfel
beantwortet nur die Fragen, auf die er gebaut wurde.

In [ ]:
if SILBER:
    delikte, familien, tage, tracts = con.execute(f"""
        SELECT count(DISTINCT nibrs_code), count(DISTINCT left(nibrs_code,2)),
               count(DISTINCT occurred_date_local), count(DISTINCT tract_geoid)
        FROM {SILBER}""").fetchone()
    print(f'In den Falldaten: {delikte} Delikte, {familien} Familien, '
          f'{tage} Tage, {tracts} Tracts')
    print()
    print(f"{'Tabelle':44}{'Zeilen':>10}")
    for name, sql in (
        ('fact_tract_month (geoid x Monat x Familie)', f"""
            SELECT count(*) FROM (SELECT tract_geoid,
              date_trunc('month',occurred_date_local), left(nibrs_code,2)
              FROM {SILBER} WHERE counts GROUP BY 1,2,3)"""),
        ('fact_city_day (Stadt x Tag x Familie)', f"""
            SELECT count(*) FROM (SELECT city, occurred_date_local,
              left(nibrs_code,2) FROM {SILBER} WHERE counts GROUP BY 1,2,3)"""),
        ('die Granularität behalten (eine Zeile je Tat)',
         f'SELECT count(*) FROM {SILBER} WHERE counts'),
    ):
        print(f'{name:44}{con.execute(sql).fetchone()[0]:>10,}')

### 2.1 Der Würfel, einmal wirklich gebaut

Bisher stand hier eine Tabelle mit den Zeilenzahlen, die ein Würfel *hätte*.
Das ist eine Behauptung. Die nächsten Zellen bauen beide Würfel wirklich und
stellen ihnen dieselben Fragen wie dem Stern — damit sichtbar wird, was eine
vorberechnete Tabelle kann und woran sie scheitert.

**In Gold werden sie nicht gebaut.** Gold bleibt der Stern. Seit dem 17.9.2026
gibt es in **Serving** vier aus Gold abgeleitete Vergleichstabellen
(`serving.build_comparison`: `comparison_day`, `comparison_week`,
`comparison_month`, `partition_quality`) — als Teil des
Veröffentlichungsstands, den ein Dashboard liest, und als gebauter
Gegenentwurf. Die beiden Würfel hier sind etwas anderes: die **ursprünglich
geplanten** `fact_tract_month` und `fact_city_day`, nachgebaut, um sie dem
Stern gegenüberzustellen. Sie sind ein Beleg, kein Bestand.

In [ ]:
if SILBER:
    con.execute(f"""CREATE OR REPLACE TABLE fact_tract_month AS
        SELECT tract_geoid,
               date_trunc('month', occurred_date_local) AS monat,
               left(nibrs_code, 2)                      AS familie,
               count(*)                                 AS delikte
        FROM {SILBER} WHERE counts
        GROUP BY 1, 2, 3""")
    con.execute(f"""CREATE OR REPLACE TABLE fact_city_day AS
        SELECT city,
               occurred_date_local     AS tag,
               left(nibrs_code, 2)     AS familie,
               count(*)                AS delikte
        FROM {SILBER} WHERE counts
        GROUP BY 1, 2, 3""")

    stern = con.execute(f'SELECT count(*) FROM {SILBER} WHERE counts').fetchone()[0]
    print(f"{'Tabelle':34}{'Zeilen':>10}{'gegen den Stern':>18}")
    print('-' * 62)
    for name in ('fact_tract_month', 'fact_city_day'):
        n = con.execute(f'SELECT count(*) FROM {name}').fetchone()[0]
        print(f'{name:34}{n:>10,}{n / stern:>17.1%}')
    print(f"{'der Stern (eine Zeile je Tat)':34}{stern:>10,}{'100.0%':>18}")

    print()
    summe = con.execute('SELECT sum(delikte) FROM fact_city_day').fetchone()[0]
    print(f'Gegenprobe: die Würfelzellen summieren sich auf {summe:,} Taten '
          f'- {"stimmt" if summe == stern else "STIMMT NICHT"}.')

Die Summen stimmen — ein Würfel ist nicht falsch, er ist **ärmer**.
`fact_tract_month` spart nicht einmal viel: Es ist kein Zehntel der
Faktentabelle, sondern gut ein Viertel.

Jetzt dieselben drei Fragen an alle drei Tabellen.

In [ ]:
if SILBER:
    from datetime import date
    LADEN, TAG, TRACT = '23C', date(2024, 11, 9), '17031839100'

    # 1 - ein einzelner Code statt einer Familie. 23C ist Ladendiebstahl,
    #     die Familie 23 enthaelt zehn weitere Diebstahlcodes.
    stern_code = con.execute(
        f"SELECT count(*) FROM {SILBER} WHERE counts AND nibrs_code = ?",
        [LADEN]).fetchone()[0]
    familie = con.execute(
        "SELECT sum(delikte) FROM fact_city_day WHERE familie = ?",
        [LADEN[:2]]).fetchone()[0]

    # 2 - ein einzelner Tag.
    stern_tag = con.execute(
        f"SELECT count(*) FROM {SILBER} WHERE counts AND occurred_date_local = ?",
        [TAG]).fetchone()[0]
    cd_tag = con.execute(
        "SELECT sum(delikte) FROM fact_city_day WHERE tag = ?", [TAG]).fetchone()[0]

    # 3 - ein einzelner Tract.
    stern_tract = con.execute(
        f"SELECT count(*) FROM {SILBER} WHERE counts AND tract_geoid = ?",
        [TRACT]).fetchone()[0]
    tm_tract = con.execute(
        "SELECT sum(delikte) FROM fact_tract_month WHERE tract_geoid = ?",
        [TRACT]).fetchone()[0]

    zeilen = [
        ('nur Ladendiebstahl 23C, Jahr', f'{stern_code:,}',
         'nur Familie 23', f'nur Familie 23 ({familie:,})'),
        ('der 9. November, beide Staedte', f'{stern_tag:,}',
         'nur Monate', f'{cd_tag:,}'),
        ('ein Tract, ganzes Jahr', f'{stern_tract:,}',
         f'{tm_tract:,}', 'nur Staedte'),
    ]
    print(f"{'Frage':32}{'Stern':>10}{'tract_month':>16}{'city_day':>26}")
    print('-' * 84)
    for frage, a, b, c in zeilen:
        print(f'{frage:32}{a:>10}{b:>16}{c:>26}')
    print()
    print('Jede Frage braucht einen anderen Wuerfel - oder den Stern, der alle drei kann.')

**Der faire Gegentest.** Ein Würfel ist nicht nutzlos: Für genau die Frage,
auf die er gebaut wurde, ist er ein Nachschlagen statt einer Rechnung. Die
nächste Zelle misst beides gegeneinander, damit die Entscheidung nicht auf
einer ungeprüften Annahme ruht.

In [ ]:
if SILBER:
    frage_stern = f"""SELECT count(*) FROM {SILBER}
        WHERE counts AND city = 'chicago'
          AND left(nibrs_code, 2) = '23'
          AND date_trunc('month', occurred_date_local) = DATE '2024-06-01'"""
    frage_wuerfel = """SELECT sum(delikte) FROM fact_city_day
        WHERE city = 'chicago' AND familie = '23'
          AND date_trunc('month', tag) = DATE '2024-06-01'"""

    a, _ = miss(frage_stern)
    b, _ = miss(frage_wuerfel)
    print(f'dieselbe Frage auf dem Stern   : {a:6.2f} ms')
    print(f'dieselbe Frage auf dem Wuerfel : {b:6.2f} ms')
    print(f'Ersparnis                      : {a - b:6.2f} ms')
    print()
    print('Beide Werte liegen unter der Wahrnehmungsschwelle eines Menschen')
    print('(etwa 100 ms). Der Wuerfel gewinnt - und der Gewinn ist unsichtbar.')

**Das ist die ganze Rechnung.** Der Würfel *ist* schneller — um ein
Vielfaches, nur liegt dieses Vielfache im Bereich von Zehntelmillisekunden und
damit weit unter allem, was jemand bemerkt. Dafür kostet er drei Fragen aus der
Tabelle oben, einen zweiten Bauauftrag und eine zweite Stelle, an der Zahlen
auseinanderlaufen können.

Bei zweihundert Millionen Zeilen sähe die Rechnung anders aus — dann wäre der
Würfel richtig, und er liesse sich jederzeit *aus* dem Stern erzeugen. Der
umgekehrte Weg existiert nicht: Was in der Zelle zusammengefasst wurde, ist
nicht versteckt, sondern weg.

> **Für den Leistungsnachweis.** Beide Bauweisen stehen hier gebaut
> nebeneinander, mit derselben Gegenprobe und derselben Messung. Die
> Entscheidung für den Stern ist damit kein Verzicht auf den Würfel, sondern
> ein Ergebnis — und das ist der Teil, der sich nicht abschreiben lässt.

**Gold behält deshalb die Granularität** und wird statt dessen *abfragbar* gemacht: eine
Faktentabelle je Delikt, vier kleine Dimensionen daneben.

### 2.2 Ein Stellenwechsel bleibt: die Tatebene

Silver hält **jede Anklage**. Detroit liefert zu einem Vorfall mehrere Zeilen —
ein Überfall mit Schüssen und geraubtem Auto steht dort als fünf, bei Chicago
als eine. Die nächste Zelle zeigt, was das ausmacht.

In [ ]:
_silber = DATA_ROOT / 'silver' / 'dataset=crime_offense' / 'part-0001.parquet'
if _silber.exists():
    S = f"read_parquet('{_silber}')"
    print(f"{'Stadt':10}{'Zeilen (Anklagen)':>20}{'zaehlend (Taten)':>20}{'Differenz':>12}")
    for stadt, alle, zaehlend in con.execute(f'''
            SELECT city, count(*), count(*) FILTER (counts)
            FROM {S} GROUP BY 1 ORDER BY 1''').fetchall():
        d = f'{(zaehlend-alle)/alle:+.1%}' if alle != zaehlend else '-'
        print(f'{stadt:10}{alle:>20,}{zaehlend:>20,}{d:>12}')

    print()
    print('Warum eine Zeile zaehlt - oder nicht:')
    for grund, zaehlt, n in con.execute(f'''
            SELECT counts_reason, counts, count(*) FROM {S}
            GROUP BY 1,2 ORDER BY 3 DESC''').fetchall():
        print(f'   {grund:34}{str(zaehlt):7}{n:>9,}')

## 3. Die Stadtzuordnung — der zweite Grund für diese Schicht

Bis hierher konnte das Projekt **keine Rate je Einwohner belegen**.
`silver/tract` deckt ganze Countys ab, und ein Tract sagt nicht, zu welcher
Stadt er gehört. Cook County hat 5,18 Millionen Einwohner, Chicago rund 2,7
Millionen — wer den County als Nenner nimmt, halbiert jede Rate.

`silver/place` liefert jetzt die Stadtgrenzen aus TIGER/Line. Damit stellt sich
die eigentliche Frage: **Tracts liegen nicht sauber in Stadtgrenzen.** Sie
überschreiten sie. Drei Regeln wären denkbar, und die nächste Zelle rechnet
alle drei.

In [ ]:
_tract = DATA_ROOT / 'silver' / 'dataset=tract' / 'vintage=2024' / 'part-0001.parquet'
_place = DATA_ROOT / 'silver' / 'dataset=place' / 'vintage=2024' / 'part-0001.parquet'

if _tract.exists() and _place.exists():
    con.execute('INSTALL spatial'); con.execute('LOAD spatial')
    # STADT_SCHLUESSEL, nicht STUDIENSTAEDTE: Beide bilden dieselbe GEOID
    # ab, aber der eine auf 'chicago' und der andere auf 'Chicago'. Diese
    # Zelle vergleicht gegen Werte aus Silver und Gold, und die sind
    # kleingeschrieben.
    from gold_pipeline import STADT_SCHLUESSEL, STADT_SCHWELLE   # noqa: E402

    codes = ', '.join(f"('{g}','{s}')" for g, s in sorted(STADT_SCHLUESSEL.items()))
    con.execute(f"""CREATE OR REPLACE TABLE anteil AS
      WITH stadt(city_geoid, city) AS (VALUES {codes}),
      grenze AS (
        SELECT s.city, ST_GeomFromWKB(p.geometry) AS geom
        FROM read_parquet('{_place}') p JOIN stadt s ON s.city_geoid = p.geoid
      )
      SELECT t.geoid, t.population, g.city,
             ST_Area(ST_Intersection(ST_GeomFromWKB(t.geometry), g.geom))
               / nullif(ST_Area(ST_GeomFromWKB(t.geometry)),0) AS flaechenanteil
      FROM read_parquet('{_tract}') t
      JOIN grenze g ON ST_Intersects(ST_GeomFromWKB(t.geometry), g.geom)""")

    print('Tracts, die eine Stadtgrenze beruehren:')
    for r in con.execute("""
        SELECT city, count(*) beruehrt,
               count(*) FILTER (WHERE flaechenanteil > 0.999) ganz_drin,
               count(*) FILTER (WHERE flaechenanteil BETWEEN 0.001 AND 0.999) geteilt,
               count(*) FILTER (WHERE flaechenanteil <= 0.001) nur_beruehrt
        FROM anteil GROUP BY 1 ORDER BY 1""").fetchall():
        print(f'  {r[0]:9} beruehrt {r[1]:>4}, ganz drin {r[2]:>4}, '
              f'geteilt {r[3]:>3}, nur beruehrt {r[4]:>3}')

    print()
    # Abgeleitet, nicht eingetippt. Hier standen 'chicago' und 'detroit'
    # fest im Code. Als am 14.9.2026 die doppelte Zuordnung entflochten
    # wurde (9d287a1), importierte diese Zelle weiter STUDIENSTAEDTE -
    # jetzt mit grossgeschriebenen Werten - und brach mit einem KeyError.
    # Gesehen hat es niemand: Der Datenlauf jenes Tages lief um 06:05 UTC,
    # der Commit kam um 12:34 UTC, und der naechste Lauf ist montags.
    staedte = sorted(STADT_SCHLUESSEL.values())
    zensus_2020 = {'chicago': 2_746_388, 'detroit': 639_111}

    print(f"{'Regel':28}" + ''.join(f'{s:>12}' for s in staedte))
    regeln = [
        ('alle beruehrten Tracts', 'sum(population)'),
        ('Flaechengewichtung', 'round(sum(population*flaechenanteil))'),
        (f'Mehrheitsregel (>{STADT_SCHWELLE})',
         f'sum(population) FILTER (WHERE flaechenanteil > {STADT_SCHWELLE})'),
    ]
    for name, ausdruck in regeln:
        werte = dict(con.execute(
            f'SELECT city, {ausdruck} FROM anteil GROUP BY 1').fetchall())
        fehlend = [s for s in staedte if s not in werte]
        if fehlend:
            raise AssertionError(f'keine Flaeche gefunden fuer {fehlend} - '
                                 f'gemessen wurde {sorted(werte)}')
        print(f'{name:28}' + ''.join(f'{int(werte[s]):>12,}' for s in staedte))
    print(f"{'Zensus 2020':28}"
          + ''.join(f'{zensus_2020[s]:>12,}' for s in staedte))

**Die naive Regel ist unbrauchbar.** „Alle berührten Tracts" zählt Chicago um
16 und Detroit um 38 Prozent zu hoch — sie schluckt jeden Nachbarort, der eine
Grenze streift. Bei Detroit sind das 78 Tracts, die die Stadtgrenze nur
berühren, ohne nennenswert in ihr zu liegen.

**Gewählt ist die Mehrheitsregel:** Ein Tract gehört zu der Stadt, die mehr als
die Hälfte seiner Fläche bedeckt. Zwei Gründe:

- Sie gibt jedem Tract **genau eine** Stadt. Das ist die Voraussetzung dafür,
  dass sich Fälle ohne Doppelzählung summieren lassen — bei einer
  Flächengewichtung müsste jeder Fall anteilig auf zwei Städte verteilt werden,
  was bei Fallzahlen keinen Sinn ergibt.
- Sie liegt praktisch gleichauf mit der Flächengewichtung. Der Unterschied ist
  klein genug, dass die einfachere Regel nichts kostet.

**Detroit trifft die amtliche Zahl auf unter ein Prozent.** Der grössere Abstand
bei Chicago ist keine Ungenauigkeit der Regel, sondern der Unterschied zwischen
einer ACS-Fünfjahresschätzung und einer Zählung.

`dim_tract` hält den Flächenanteil trotzdem in jeder Zeile fest — wer
flächengewichtet rechnen will, kann es, und wer die Regel prüfen will, auch.

### 3.1 Was die Regel an den Rändern kostet

Eine Zuordnung nach Mehrheit heisst: Fälle in Tracts, die *knapp* ausserhalb
liegen, fallen aus dem Nenner. Die nächste Zelle misst, wie viele das sind.

In [ ]:
if 'fact_offense' not in fehlend:
    print('Delikte nach meldender Stadt und Stadt des Tracts:')
    for r in con.execute("""
        SELECT f.reporting_city, coalesce(t.city, '(keine)') AS tract_stadt,
               count(*) AS n, round(avg(t.city_area_share), 3) AS anteil
        FROM fact_offense f JOIN dim_tract t ON t.geoid = f.tract_geoid
        GROUP BY 1, 2 ORDER BY 1, 3 DESC""").fetchall():
        print(f'  meldend {r[0]:9} Tract {r[1]:9} {r[2]:>8,}  '
              f'mittlerer Flaechenanteil {r[3]}')

    gesamt, verortet, ohne_stadt = con.execute("""
        SELECT count(*), count(f.tract_geoid),
               count(*) FILTER (WHERE t.city IS NULL AND f.tract_geoid IS NOT NULL)
        FROM fact_offense f LEFT JOIN dim_tract t ON t.geoid = f.tract_geoid""").fetchone()
    print()
    print(f'Delikte insgesamt          : {gesamt:,}')
    print(f'davon in einem Tract       : {verortet:,} = {verortet/gesamt:.1%}')
    print(f'davon in keiner Studienstadt: {ohne_stadt:,} = {ohne_stadt/gesamt:.1%}')

**Rund 98 Prozent der Taten liegen in einem Tract der meldenden Stadt.** Der
Rest sitzt an der Grenze — Fälle, die eine Stadtpolizei knapp jenseits der
Stadtgrenze erfasst hat, und Tracts, die zu weniger als der Hälfte in der Stadt
liegen.

Das ist der Preis der Regel, und er ist **gemessen statt geschätzt**. Für eine
Rate je Einwohner ist er vertretbar: Zähler und Nenner folgen derselben Grenze,
also verzerrt der Rand beide Seiten gleichsinnig.

### 3.2 Die Gegenprobe: stimmt die Stadtzuordnung überhaupt?

Kapitel 9 nannte lange eine Grenze: *„Der räumliche Join ist nicht unabhängig
validiert."* Der Grund war der Befund zum räumlichen Join (8.4) — Detroits
eigene `census_block_2020_geoid`
widerspricht Detroits eigenen Koordinaten (8.4), taugt also nicht als Referenz.

Es gibt aber eine zweite, und sie lag die ganze Zeit im Projekt:
**`silver/place`, die Stadtgrenze selbst.** Die Mehrheitsregel geht den Umweg
über den Tract — ein Punkt liegt in einem Tract, und der Tract gehört zu einer
Stadt. Man kann denselben Punkt aber auch **direkt** gegen die Ortsgrenze
halten. Andere Geometrie, andere Operation, dieselbe Frage.

Stimmen beide überein, ist die Zuordnung bestätigt. Gehen sie auseinander,
sagt die Richtung, welchen Fehler die Regel macht.

In [ ]:
if 'fact_offense' not in fehlend and _place.exists():
    con.execute(f"""CREATE OR REPLACE VIEW ortsgrenze AS
        SELECT geoid, name, ST_GeomFromWKB(geometry) AS geom
        FROM read_parquet('{_place}')
        WHERE geoid IN ({', '.join(f"'{g}'" for g in sorted(STADT_SCHLUESSEL))})""")

    print('Stadt laut Mehrheitsregel gegen Stadt laut direkter Ortsgrenze:')
    print()
    for r in con.execute("""
        SELECT coalesce(t.city, '(keine)')            AS regel,
               coalesce(lower(o.name), '(ausserhalb)') AS direkt,
               count(*)                               AS n
        FROM fact_offense f
        LEFT JOIN dim_tract t ON t.geoid = f.tract_geoid
        LEFT JOIN ortsgrenze o
          ON ST_Within(ST_Point(f.longitude, f.latitude), o.geom)
        WHERE f.latitude IS NOT NULL
        GROUP BY 1, 2 ORDER BY 3 DESC""").fetchall():
        zeichen = '  ' if r[0] == r[1] else '<-'
        print(f'  {zeichen} Regel {r[0]:<10} direkt {r[1]:<13} {r[2]:>8,}')

**Kein einziger Fall wird der falschen Stadt zugeordnet.** Es gibt keine Zeile,
in der die Regel *Chicago* sagt und die Ortsgrenze *Detroit* — oder umgekehrt.
Das ist die Zusage, auf der jeder Städtevergleich ruht, und sie ist damit
unabhängig bestätigt.

Die Abweichungen liegen alle in einer Richtung: Die Regel **schliesst aus**,
was drin ist, und schliesst fast nie ein, was draussen ist. Genau ein Fall
(von 251'516) liegt in einem Chicago-Tract und trotzdem knapp jenseits der
Stadtgrenze.

Die nächste Zelle nimmt sich die andere Richtung vor — die Fälle, die die
Regel verliert.

In [ ]:
if 'fact_offense' not in fehlend and _place.exists():
    print('Die Faelle ausserhalb: wo liegen sie wirklich?')
    print()
    for r in con.execute("""
        SELECT f.reporting_city,
               CASE
                 WHEN o.geoid IS NOT NULL THEN 'in der meldenden Stadt'
                 WHEN n.name IS NOT NULL  THEN 'in einer Nachbargemeinde'
                 ELSE 'in keiner Gemeinde'
               END AS lage,
               count(*) AS n
        FROM fact_offense f
        JOIN dim_tract t ON t.geoid = f.tract_geoid
        LEFT JOIN ortsgrenze o
          ON ST_Within(ST_Point(f.longitude, f.latitude), o.geom)
        LEFT JOIN (SELECT name, ST_GeomFromWKB(geometry) AS geom
                   FROM read_parquet(?)) n
          ON o.geoid IS NULL
         AND ST_Within(ST_Point(f.longitude, f.latitude), n.geom)
        WHERE t.city IS NULL
        GROUP BY 1, 2 ORDER BY 1, 3 DESC""", [str(_place)]).fetchall():
        print(f'  {r[0]:<9} {r[1]:<26} {r[2]:>6,}')

Damit sind die Fälle ohne Stadtzuordnung **klassifiziert** statt bloss gezählt.
Sie zerfallen in zwei Gruppen mit ganz verschiedener Bedeutung:

* **In der meldenden Stadt.** Der Punkt liegt innerhalb der Stadtgrenze, aber
  sein Tract gehört mehrheitlich zur Nachbargemeinde. Das ist ein Fehler
  **der Regel**, kein Fehler der Daten — und es sind über 5'000 Fälle.
* **In einer Nachbargemeinde** (Oak Park, Evanston, Warren, Eastpointe …).
  Die Stadtpolizei hat jenseits der Stadtgrenze erfasst. Diese Fälle gehören
  zu Recht nicht in den Zähler.

Die dritte, kleine Gruppe liegt in **keiner** Gemeinde — nicht eingemeindetes
Gebiet zwischen den Orten.

Bleibt die Frage, die wirklich zählt: **Verzerrt das die Rate?**

In [ ]:
if 'fact_offense' not in fehlend and _place.exists():
    print(f"{'Stadt':9}{'Delikte':>10}{'Einwohner':>12}{'Rate':>7}   Variante")
    # STADT_SCHLUESSEL: Der Wert geht unten in "WHERE t.city = ?", und
    # dim_tract.city ist kleingeschrieben. Mit STUDIENSTAEDTE haette diese
    # Zelle nicht gescheitert, sondern stillschweigend null Zeilen gezaehlt.
    for stadt, ort_geoid in sorted(
        (s, g) for g, s in STADT_SCHLUESSEL.items()
    ):
        drin, einwohner = con.execute(f"""
            SELECT count(*), any_value(e.n)
            FROM fact_offense f JOIN dim_tract t ON t.geoid = f.tract_geoid
            JOIN ({NENNER}) e ON e.city = t.city
            WHERE t.city = ?""", [stadt]).fetchone()
        # Was die Regel verliert - Zaehler und Nenner zusammen, sonst waere
        # der Vergleich unehrlich: Die Einwohner dieser Tracts fehlen ebenso.
        # Die Einwohnerzahl kommt aus dim_tract_acs, am Referenz-Vintage der
        # Stadtzuordnung - demselben Stand, dessen Regel hier geprueft wird.
        verloren, nenner = con.execute("""
            WITH betroffen AS (
                SELECT f.offense_id, t.geoid, a.population, t.city_area_share
                FROM fact_offense f
                JOIN dim_tract t ON t.geoid = f.tract_geoid
                JOIN dim_tract_acs a
                  ON a.geoid = t.geoid AND a.vintage = t.reference_vintage
                JOIN ortsgrenze o ON o.geoid = ?
                 AND ST_Within(ST_Point(f.longitude, f.latitude), o.geom)
                WHERE t.city IS NULL
            )
            SELECT count(*),
                   (SELECT round(sum(population * city_area_share))
                    FROM (SELECT DISTINCT geoid, population, city_area_share
                          FROM betroffen))
            FROM betroffen""", [ort_geoid]).fetchone()
        nenner = nenner or 0
        print(f'  {stadt:<7}{drin:>10,}{einwohner:>12,}'
              f'{drin * 1000 / einwohner:>7.1f}   Mehrheitsregel (gebaut)')
        if verloren:
            print(f"  {'':<7}{drin + verloren:>10,}{einwohner + nenner:>12,.0f}"
                  f'{(drin + verloren) * 1000 / (einwohner + nenner):>7.1f}   '
                  f'mit den {verloren:,} Faellen an der Grenze')

**Die Randtracts sind überdurchschnittlich deliktdicht.** Die 5'249 Chicagoer
Fälle sitzen auf rund 23'100 anteiligen Einwohnern — das sind **227 Delikte je
1'000**, gegen 94,5 im Stadtdurchschnitt. Plausibel: Es sind Gewerbestreifen,
Verkehrsachsen und Flughafenumland, wo viele Menschen sind, aber wenige wohnen.

Deshalb ist die Regel **nicht neutral**. Chicagos Rate liegt mit ihr bei 94,5
statt 95,6 — sie ist um gut einen Punkt **untertrieben**. Detroit ist praktisch
nicht betroffen (12 Fälle, keine Einwohner).

**Was das für den Städtevergleich heisst.** Der Abstand schrumpft leicht: 132,7
gegen 94,5 sind 40 % mehr, gegen 95,6 noch 39 %. Die Aussage bleibt, ihre
Grösse verschiebt sich um gut einen Prozentpunkt.

**Warum die Regel trotzdem bleibt.** Eine Zuordnung je Punkt statt je Tract
würde den Zähler genauer machen und den Nenner unmöglich: Einwohnerzahlen gibt
es nur je Tract, nicht je Punkt. Man bekäme eine exakte Zahl oben und eine
geschätzte unten — und das ist schlechter als zwei Zahlen, die derselben Regel
folgen. Die Verzerrung ist jetzt gemessen und benannt, statt unbekannt zu sein.

## 4. Die Rate, die es vorher nicht gab

Damit lässt sich zum ersten Mal beantworten, was das Projekt von Anfang an
fragen wollte — und was mit absoluten Fallzahlen nicht zu beantworten war,
weil Chicago rund viermal so viele Einwohner hat wie Detroit.

In [ ]:
if 'fact_offense' not in fehlend:
    # Einmal ausgeschrieben: So sieht der Nenner aus, seit die Einwohnerzahl je
    # Vintage liegt. Das zweite Praedikat im Verbund ist der ganze Unterschied.
    print('Der Nenner, aus gold_pipeline.einwohner_je_stadt:')
    print('   ' + ' '.join(NENNER.split()))
    print()

    for r in con.execute(f"""
        WITH einwohner AS ({NENNER})
        SELECT t.city, count(*) AS delikte, any_value(e.n) AS einwohner,
               round(count(*) * 1000.0 / any_value(e.n), 1) AS je_1000
        FROM fact_offense f
        JOIN dim_tract t ON t.geoid = f.tract_geoid
        JOIN einwohner e ON e.city = t.city
        WHERE t.city IS NOT NULL GROUP BY 1 ORDER BY 1""").fetchall():
        print(f'  {r[0]:9} {r[1]:>8,} Delikte  {r[2]:>10,} Einwohner  '
              f'{r[3]:>6} je 1000')

    print()
    print('Die groessten Deliktfamilien, je 1000 Einwohner:')
    print(f"  {'Familie':9}{'Chicago':>10}{'Detroit':>10}{'Verhaeltnis':>13}")
    for r in con.execute(f"""
        WITH einwohner AS ({NENNER}),
        rate AS (
            SELECT t.city, f.nibrs_family,
                   count(*) * 1000.0 / any_value(e.n) AS je_1000
            FROM fact_offense f
            JOIN dim_tract t ON t.geoid = f.tract_geoid
            JOIN einwohner e ON e.city = t.city
            WHERE t.city IS NOT NULL GROUP BY 1, 2
        )
        SELECT nibrs_family,
               round(max(je_1000) FILTER (WHERE city='chicago'), 1),
               round(max(je_1000) FILTER (WHERE city='detroit'), 1)
        FROM rate GROUP BY 1
        HAVING max(je_1000) FILTER (WHERE city='chicago') > 3
           AND max(je_1000) FILTER (WHERE city='detroit') > 3
        ORDER BY 2 DESC""").fetchall():
        v = f'{r[2]/r[1]:.2f}x' if r[1] else '-'
        print(f'  {r[0]:9}{r[1]:>10}{r[2]:>10}{v:>13}')

**Detroit liegt je Einwohner deutlich höher als Chicago** — und das ist die
Aussage, die absolute Fallzahlen genau umgekehrt nahegelegt hätten, weil
Chicago in jeder Kategorie mehr Fälle zählt.

Zwei Vorbehalte gehören dazu:

- **Anzeigeverhalten ist nicht Kriminalität.** Beide Quellen zählen gemeldete
  Delikte, und die Meldebereitschaft ist zwischen zwei Städten nicht konstant.
- **Der Nenner ist eine ACS-Schätzung**, keine Zählung, und die Stadtzuordnung
  ist die Mehrheitsregel aus Abschnitt 3.

## 5. Die Fragen der Auswertungsplattform, gemessen

Zum Schluss die Probe aufs Ganze: genau die Abfragen, für die das Sternschema
gebaut ist — ein Delikt auswählen, dann Tag, Monat, Jahr; oder nur eine Stadt.

In [ ]:
if 'fact_offense' not in fehlend:
    fragen = [
        ('ein Delikt + ein Tag + eine Stadt', """
            SELECT count(*) FROM fact_offense f JOIN dim_offense o USING (nibrs_code)
            WHERE o.nibrs_description = 'Shoplifting'
              AND f.date = DATE '2024-11-09' AND f.reporting_city = 'chicago'"""),
        ('ein Delikt + ein Monat, beide Staedte', """
            SELECT f.reporting_city, count(*) FROM fact_offense f
            JOIN dim_date d USING (date)
            WHERE f.nibrs_code = '240' AND d.month = 6 GROUP BY 1"""),
        ('nur eine Stadt, Delikte je Monat', """
            SELECT d.month, count(*) FROM fact_offense f JOIN dim_date d USING (date)
            WHERE f.reporting_city = 'detroit' GROUP BY 1 ORDER BY 1"""),
        ('Rate je 1000 Einwohner, Stadt x Delikt', f"""
            WITH e AS ({NENNER})
            SELECT t.city, f.nibrs_code, count(*) * 1000.0 / any_value(e.n)
            FROM fact_offense f JOIN dim_tract t ON t.geoid = f.tract_geoid
            JOIN e ON e.city = t.city WHERE t.city IS NOT NULL GROUP BY 1, 2"""),
        ('Wochenende gegen Werktag, je Familie', """
            SELECT f.nibrs_family, d.is_weekend, count(*) FROM fact_offense f
            JOIN dim_date d USING (date) GROUP BY 1, 2"""),
        ('warme gegen kalte Tage, eine Stadt', """
            SELECT w.temp_max_c > 25 AS warm, count(*) FROM fact_offense f
            JOIN dim_city_day w ON w.city = f.reporting_city AND w.date = f.date
            WHERE f.reporting_city = 'chicago' GROUP BY 1"""),
    ]
    print(f"{'Abfrage auf Gold':40}{'ms':>8}{'Zeilen':>9}")
    for name, sql in fragen:
        ms, zeilen = miss(sql)
        print(f'{name:40}{ms:>8.1f}{zeilen:>9,}')

**Jede Frage ist ein Filter und ein `GROUP BY` auf einer Tabelle.** Die
Oberfläche baut die `WHERE`-Klausel aus den Auswahlfeldern; mehr braucht es
nicht. Kein Würfel muss vorher wissen, welche Kombination jemand anklickt.

**Und die Dimensionen tragen die Auswahllisten gleich mit:** `dim_offense`
liefert die Delikte und ihre Familien, `dim_date` Monate und Wochentage,
`dim_tract` die Städte. Ein Auswahlfeld ist damit ein `SELECT DISTINCT` auf
eine Dimension und keine gepflegte Liste im Frontend.

## 5.1 Zurückgezogene Codes müssen auflösbar sein

Vier NIBRS-Codes hat das FBI gestrichen; die Fassung 2025 kennt sie nicht mehr.
Die MICR-Quellen führen sie weiter, und **zwei stehen in den Falldaten 2024**.

Sie einfach wegzulassen wäre der schlechteste Weg — dann verlöre jeder Filter
über die Dimension rund tausend Delikte, ohne zu scheitern. Sie nur als
`retired` zu markieren ist aber auch zu wenig: Das sagt, dass der Code weg ist,
nicht **wohin**. Wer eine Zeitreihe über mehrere Jahrgänge legt oder nach einem
Delikt filtert, übersieht sonst bei `26A` genau die Fälle, die früher `90A`
hiessen.

`dim_offense` trägt deshalb drei weitere Spalten: `resolution`, `replaced_by`
und `retirement_note`. Aufgelöst wird nur, wo es einen **Beleg** gibt — die
Spezifikation schweigt, sie kennt die Codes gar nicht mehr. Belegen lässt es
sich über die ILCS-Tabelle der Illinois State Police, dieselbe Quelle, die
schon die Chicagoer Gegenprobe trägt.

In [ ]:
if 'dim_offense' not in fehlend:
    print(f"{'Code':6}{'Bezeichnung':16}{'Aufloesung':18}{'heute':8}{'Faelle':>8}")
    for r in con.execute("""
        SELECT d.nibrs_code, d.nibrs_description, d.resolution,
               coalesce(d.replaced_by, '-') AS heute,
               count(f.offense_id) AS faelle
        FROM dim_offense d
        LEFT JOIN fact_offense f ON f.nibrs_code = d.nibrs_code
        WHERE d.retired GROUP BY 1, 2, 3, 4 ORDER BY 1""").fetchall():
        print(f'{r[0]:6}{r[1]:16}{r[2]:18}{r[3]:8}{r[4]:>8,}')

    print()
    print('Die Begruendungen:')
    for code, grund in con.execute(
        "SELECT nibrs_code, retirement_note FROM dim_offense "
        "WHERE retired ORDER BY nibrs_code"
    ).fetchall():
        print(f'  {code}: {grund}')

**`90A Bad Checks` löst sich auf `26A` auf**, und das ist belegt: Die
ILCS-Tabelle führt `720-5/17-1(B)` *Deceptive Practices – Bad Checks* unter
`26A False Pretenses/Swindle/Confidence Game`. Detroits MICR-Zeile `2606`
*Fraud – Bad Checks* liegt in derselben Deliktfamilie `26`.

**`90I Runaway` löst sich auf gar nichts auf** — und genau das ist die Antwort.
Weglaufen ist keine Straftat; deshalb hat das FBI den Code gestrichen. Die
ILCS-Tabelle kennt nur `720-5/10-6 Harboring a Runaway`, das *Beherbergen*, und
ordnet es `90Z` zu. Die 967 Zeilen bleiben zählbar, tragen aber kein heutiges
NIBRS-Delikt.

**`90E` und `90H` bleiben offen.** Öffentliche Trunkenheit ist in Illinois keine
Straftat, und die nächstliegenden Statute zum Spannertum
(`720-5/26-4 Unauthorized Video Recording`) verteilen sich je nach Ort auf `370`
und `90C` — kein einzelner Nachfolger. Beide kommen in den Falldaten 2024 nicht
vor, und **genau das erzwingt der Bau**: Ein Code mit
`resolution = 'unresolved'`, der in den Fakten auftaucht, lässt `reconcile`
fallen. Erscheint er in einem späteren Jahrgang, muss jemand einen Beleg
nachtragen, statt ihn stillschweigend mitlaufen zu lassen.

## 5.2 Die Geo-Lücke auf gemeinsamer Achse

Beide Städte verlieren Fälle ohne Koordinate, und in beiden ist der Ausfall
**nicht zufällig** — das ist der Befund zu den fehlenden Koordinaten
(Konzept 8.7), je Stadt gemessen in Notebook 03 und 13.

Dort steht die Frage aber je Stadt in **ihren eigenen** Kategorien: Chicago
kennt „Delikte an Kindern", Detroit „Family Offense". Ob dieselbe Ursache
wirkt, lässt sich so nicht sagen. Erst `dim_offense.nibrs_category` legt beide
auf eine Achse — und genau dafür gibt es die Dimension.

Verglichen wird nicht der Prozentsatz, sondern das **Vielfache des
Stadtschnitts**: Chicago verliert insgesamt 0,81 %, Detroit 0,29 %, und eine
Kategorie ist erst dann auffällig, wenn sie über dem eigenen Schnitt liegt.

In [ ]:
if 'dim_offense' not in fehlend and SILBER:
    print(f"{'NIBRS-Kategorie':34}{'Chicago':>18}{'Detroit':>18}")
    print(f"{'':34}{'Anteil':>9}{'Faktor':>9}{'Anteil':>9}{'Faktor':>9}")
    for r in con.execute(f"""
        WITH basis AS (
            SELECT s.city, o.nibrs_category AS kat, count(*) AS n,
                   count(*) FILTER (WHERE s.latitude IS NULL) AS ohne
            FROM {SILBER} s
            LEFT JOIN dim_offense o ON o.nibrs_code = s.nibrs_code
            GROUP BY 1, 2
        ), schnitt AS (
            SELECT city, sum(ohne) * 1.0 / sum(n) AS mittel FROM basis GROUP BY 1
        ), je_stadt AS (
            SELECT b.kat, b.city, b.n,
                   b.ohne * 1.0 / b.n           AS anteil,
                   (b.ohne * 1.0 / b.n) / s.mittel AS vielfaches
            FROM basis b JOIN schnitt s ON s.city = b.city
        )
        SELECT kat,
               max(CASE WHEN city='chicago' THEN anteil END)     AS chi_anteil,
               max(CASE WHEN city='chicago' THEN vielfaches END) AS chi_x,
               max(CASE WHEN city='detroit' THEN anteil END)     AS det_anteil,
               max(CASE WHEN city='detroit' THEN vielfaches END) AS det_x
        FROM je_stadt GROUP BY 1
        -- unter 200 Zeilen ist ein Prozentsatz Zufall, kein Befund
        HAVING max(CASE WHEN city='chicago' THEN n END) >= 200
           AND max(CASE WHEN city='detroit' THEN n END) >= 200
        ORDER BY greatest(chi_x, det_x) DESC LIMIT 6
    """).fetchall():
        print(f'{r[0][:33]:34}{r[1]:>9.2%}{r[2]:>8.1f}x{r[3]:>9.2%}{r[4]:>8.1f}x')

**Eine Kategorie trägt in beiden Städten**, und es ist dieselbe:
**Sexualdelikte**, in Chicago beim 12,0-Fachen des Stadtschnitts, in Detroit
beim 5,2-Fachen. Die Ursache liegt nahe und ist in beiden Systemen dieselbe:
Bei diesen Delikten wird die Adresse zurückgehalten. Weil die Richtung
übereinstimmt, verzerrt dieser Teil der Lücke den Städtevergleich **nicht**
systematisch — er trifft beide Seiten gleich.

**Eine zweite Kategorie trägt nur in Chicago**, und das ist der eigentliche
Befund dieses Abschnitts: **Betrug** fehlt in Chicago bei 3,78 % der Fälle
(4,7-Faches des Schnitts), in Detroit bei 0,19 % (0,7-Faches) — ein Faktor 20
zwischen den Städten. Plausibel ist, dass Chicago Betrug ohne Tatort mitführt,
den es telefonisch oder online gab; Detroits Erfassung über den Vorfallsbericht
verlangt eine Adresse.

**Was daraus für die Auswertung folgt.** Ein Vergleich der *Gesamtraten* trägt:
0,81 % gegen 0,29 % Ausfall verschieben keine Rangfolge. Ein Vergleich **je
Delikttyp** trägt für Sexualdelikte, weil die Verzerrung gleichgerichtet ist —
aber **nicht für Betrug**. Dort ist Chicagos Zahl die untererfasste, und wer
die beiden Städte bei Fraud gegeneinanderstellt, muss die Abdeckung
danebenstellen.

Das ist die Antwort auf die Frage, mit der Notebook 13 begonnen wurde: Der
Detroiter Geoausfall ist *nicht* durchweg wie der Chicagoer. Er ist es bei der
Kategorie, bei der man es vermutet — und bei einer anderen gerade nicht.

## 6. Grenzen

- **Die Familie bleibt ein Kürzel.** `dim_offense` führt `nibrs_family` als
  `13`, `23`, `26` ohne Namen — die Spezifikation führt dafür keine auslesbare
  Liste. Das ist seit dem Kategorien-Werkzeug aber kein Mangel mehr: Gruppiert
  wird über `nibrs_category`. Die Datei führt **42** Kategorien — 38 aus dem
  *User Manual 2025.0* (28 Group A, 10 Group B) und vier weitere aus der
  Codeliste 2011, die nur die vier zurückgezogenen Codes tragen (*Bad Checks*,
  *Drunkenness*, *Peeping Tom*, *Runaway*). Die Spalte `source` sagt je Zeile,
  welches Dokument sie belegt. Die Familie ist nur noch ein technisches Präfix, kein Auswertungsweg.
- **Ein Jahrgang.** Gold deckt 2024 ab. `dim_date` wird aus der Spanne der
  Fakten erzeugt und wächst mit.
- **Die Rate ist eine Näherung**, siehe Abschnitt 3: Mehrheitsregel,
  ACS-Schätzung, und rund zwei Prozent Randfälle.

In [ ]:
con.close()
print('Verbindung geschlossen.')